# 🗞️ Fantasy News Daily Refresh Pipeline

**Automated daily job** to keep fantasy football insights fresh and actionable.

---

## Overview

This notebook runs daily to ingest NFL news, extract player mentions, generate AI-powered fantasy insights, and create consolidated player notes for the FantasAI platform.

---

## Pipeline Phases

### 🔄 Phase 1: News Ingestion
- Fetch articles from fantasy news APIs (Sleeper, Rotoworld, ESPN, Yahoo)
- Scrape top NFL team RSS feeds (KC, DAL, BUF, PHI, SF, DET, BAL, CIN)
- Deduplicate by content hash
- Write to `main.fantasai_news.raw_rss_articles`

### 🔍 Phase 2: Player Entity Extraction
- Load player roster from Unity Catalog
- Pattern-match player names in article text
- Calculate confidence scores
- Write to `main.fantasai_news.enriched_news`

### 🤖 Phase 3: AI Summarization
- Load enriched articles not yet summarized
- Call Databricks Foundation Model (Llama 3.3 70B)
- Generate fantasy insights with relevance scores
- Write to `main.fantasai_news.ai_summaries`

### 📋 Phase 4: Player Notes Aggregation
- Group AI summaries by player
- Aggregate top 5 latest news items
- Calculate overall impact scores and sentiment
- Upsert to `main.fantasai_news.player_notes`

---

**Estimated Runtime:** 7-11 minutes  
**Schedule:** Daily at 7:00 AM CT  
**Output:** Updated player notes and fantasy insights

In [0]:
# Install required packages for news aggregation
print("📦 Installing dependencies...")
%pip install feedparser requests beautifulsoup4 --quiet
print("✅ Dependencies installed successfully")

In [0]:
# Import libraries
import requests
import feedparser
import pandas as pd
import json
import hashlib
import time
import re
from datetime import datetime, timedelta
from typing import List, Dict, Tuple
from pyspark.sql import functions as F
from pyspark.sql.types import *
from mlflow.deployments import get_deploy_client

print("=" * 70)
print("⚙️  FANTASY NEWS PIPELINE CONFIGURATION")
print("=" * 70)

# === NEWS SOURCE CONFIGURATION ===
# Only include working RSS feeds
FANTASY_NEWS_SOURCES = {
    "espn_fantasy_rss": "https://www.espn.com/espn/rss/fantasy/football/news"
}

# BLUESKY API - NFL Beat Reporters and Official Accounts
BLUESKY_ACCOUNTS = [
    # NFL Official
    "nfl.bsky.social",
    
    # ESPN NFL Reporters
    "adamschefter.bsky.social",
    "diannaespn.bsky.social",
    
    # NFL Network
    "rapsheet.bsky.social",  # Ian Rapoport
    "mikegarafolo.bsky.social",
    
    # Major Beat Writers (if they've moved to Bluesky)
    # Add more as they migrate
]

BLUESKY_API_BASE = "https://public.api.bsky.app"

# NFL TEAM OFFICIAL WEBSITES - All 32 Teams
NFL_TEAM_NEWS_URLS = {
    # NFC East
    "DAL": "https://www.dallascowboys.com/news",
    "NYG": "https://www.giants.com/news",
    "PHI": "https://www.philadelphiaeagles.com/news",
    "WAS": "https://www.commanders.com/news",
    
    # NFC North
    "CHI": "https://www.chicagobears.com/news",
    "DET": "https://www.detroitlions.com/news",
    "GB": "https://www.packers.com/news",
    "MIN": "https://www.vikings.com/news",
    
    # NFC South
    "ATL": "https://www.atlantafalcons.com/news",
    "CAR": "https://www.panthers.com/news",
    "NO": "https://www.neworleanssaints.com/news",
    "TB": "https://www.buccaneers.com/news",
    
    # NFC West
    "ARI": "https://www.azcardinals.com/news",
    "LAR": "https://www.therams.com/news",
    "SF": "https://www.49ers.com/news",
    "SEA": "https://www.seahawks.com/news",
    
    # AFC East
    "BUF": "https://www.buffalobills.com/news",
    "MIA": "https://www.miamidolphins.com/news",
    "NE": "https://www.patriots.com/news",
    "NYJ": "https://www.newyorkjets.com/news",
    
    # AFC North
    "BAL": "https://www.baltimoreravens.com/news",
    "CIN": "https://www.bengals.com/news",
    "CLE": "https://www.clevelandbrowns.com/news",
    "PIT": "https://www.steelers.com/news",
    
    # AFC South
    "HOU": "https://www.houstontexans.com/news",
    "IND": "https://www.colts.com/news",
    "JAX": "https://www.jaguars.com/news",
    "TEN": "https://www.titansonline.com/news",
    
    # AFC West
    "DEN": "https://www.denverbroncos.com/news",
    "KC": "https://www.chiefs.com/news",
    "LV": "https://www.raiders.com/news",
    "LAC": "https://www.chargers.com/news"
}

# Pipeline configuration
BATCH_SIZE = 50  # Process up to 50 articles per phase
MAX_RETRIES = 3
REQUEST_TIMEOUT = 10
MAX_TEAM_ARTICLES_PER_SITE = 5  # Limit articles per team to avoid overwhelming the pipeline

# Initialize Databricks Foundation Model client
try:
    llm_client = get_deploy_client("databricks")
    LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"
    print(f"✅ LLM client initialized: {LLM_MODEL}")
except Exception as e:
    llm_client = None
    print(f"⚠️  LLM client initialization failed: {e}")

print(f"\n📰 Working news sources:")
print(f"  - ESPN Fantasy RSS: 1 feed")
print(f"  - Bluesky NFL reporters: {len(BLUESKY_ACCOUNTS)} accounts")
print(f"  - NFL Team Sites: {len(NFL_TEAM_NEWS_URLS)} official team websites")
print(f"  - FootballGuys.com: Web scraper")
print(f"  - NFL.com: Web scraper")
print(f"\n⚙️  Batch size: {BATCH_SIZE} articles per phase")
print(f"⚙️  Max articles per team site: {MAX_TEAM_ARTICLES_PER_SITE}")
print(f"⏰ Pipeline started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

In [0]:
print("=" * 80)
print("🔍 FEED VALIDATION - Testing All RSS Feeds")
print("=" * 80)

import feedparser

print("\n📰 Testing Fantasy News APIs:")
print("-" * 80)
for source_name, url in FANTASY_NEWS_SOURCES.items():
    try:
        if "sleeper" in source_name:
            response = requests.get(url, timeout=REQUEST_TIMEOUT)
            status = response.status_code
            count = len(response.json()) if status == 200 else 0
            print(f"  {source_name:25} Status: {status}   Entries: {count}")
        else:
            feed = feedparser.parse(url)
            status = getattr(feed, 'status', 'N/A')
            count = len(feed.entries)
            print(f"  {source_name:25} Status: {status}   Entries: {count}")
    except Exception as e:
        print(f"  {source_name:25} ERROR: {str(e)[:40]}")

print("\n\n🏈 Testing NFL Team RSS Feeds (All 32):")
print("-" * 80)
print(f"{'Team':<6} {'Status':<8} {'Entries':<10} {'Feed URL'}")
print("-" * 80)

working_feeds = []
broken_feeds = []

for team, url in sorted(NFL_TEAM_FEEDS.items()):
    try:
        feed = feedparser.parse(url)
        status = getattr(feed, 'status', 'N/A')
        count = len(feed.entries)
        
        if count > 0:
            working_feeds.append((team, count, url))
            print(f"✅ {team:<6} {str(status):<8} {count:<10} {url[:50]}...")
        else:
            broken_feeds.append((team, status, url))
            print(f"❌ {team:<6} {str(status):<8} {count:<10} {url[:50]}...")
    except Exception as e:
        broken_feeds.append((team, 'ERROR', url))
        print(f"⚠️  {team:<6} ERROR    {str(e)[:20]:<10} {url[:50]}...")

print("\n" + "=" * 80)
print("📊 VALIDATION SUMMARY:")
print("=" * 80)
print(f"  ✅ Working feeds:  {len(working_feeds)}/32 teams")
print(f"  ❌ Broken feeds:   {len(broken_feeds)}/32 teams")
print(f"  📈 Success rate:   {len(working_feeds)/32*100:.1f}%")

if len(working_feeds) > 0:
    print(f"\n  Top working feeds:")
    for team, count, url in sorted(working_feeds, key=lambda x: x[1], reverse=True)[:5]:
        print(f"    {team}: {count} articles")

print("\n💡 RECOMMENDATION: Switch to NFL.com scraping for reliable team news")
print("=" * 80)

In [0]:
print("\n" + "=" * 70)
print("📥 PHASE 1: NEWS INGESTION")
print("=" * 70)

from bs4 import BeautifulSoup
import re

start_time = time.time()
articles_list = []

# Helper function to generate content hash
def generate_hash(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()

# === Fetch from Fantasy News APIs ===
print("\n🔄 Fetching from fantasy news APIs...")

for source_name, url in FANTASY_NEWS_SOURCES.items():
    try:
        if "sleeper" in source_name:
            # Sleeper API returns JSON
            response = requests.get(url, timeout=REQUEST_TIMEOUT)
            if response.status_code == 200:
                data = response.json()
                # Sleeper returns trending player data, not articles
                # Skip for now or adapt to create article-like entries
                print(f"  ⏭️  {source_name}: Skipped (player trending data)")
                continue
        else:
            # RSS feeds
            feed = feedparser.parse(url)
            for entry in feed.entries[:10]:  # Limit to 10 per source
                content_text = entry.get('summary', entry.get('description', ''))
                # Extract tags properly
                tags_list = [str(tag.get('term', '')) for tag in entry.get('tags', []) if tag.get('term')]
                if not tags_list:
                    tags_list = []
                
                article = {
                    "article_id": generate_hash(entry.get('link', entry.get('title', ''))),
                    "source_name": source_name,
                    "source_type": "fantasy_api",
                    "author_name": entry.get('author', None),
                    "title": entry.get('title', ''),
                    "summary": content_text[:500] if len(content_text) > 500 else content_text,
                    "full_text": content_text,
                    "article_url": entry.get('link', ''),
                    "published_at": datetime(*entry.get('published_parsed', datetime.now().timetuple())[:6]) if 'published_parsed' in entry else datetime.now(),
                    "ingested_at": datetime.now(),
                    "rss_feed_url": url,
                    "tags": tags_list,
                    "content_hash": generate_hash(content_text),
                    "is_processed": False,
                    "processed_at": None,
                    "created_at": datetime.now(),
                    "updated_at": datetime.now()
                }
                articles_list.append(article)
            print(f"  ✓ {source_name}: {len(feed.entries[:10])} articles")
    except Exception as e:
        print(f"  ✗ {source_name}: Error - {str(e)[:50]}")

# === Fetch from Bluesky API ===
print("\n🦋 Fetching from Bluesky NFL reporters...")

bluesky_posts_fetched = 0
bluesky_accounts_success = 0

for account in BLUESKY_ACCOUNTS:
    try:
        # Bluesky API endpoint to get author feed
        url = f"{BLUESKY_API_BASE}/xrpc/app.bsky.feed.getAuthorFeed"
        params = {
            "actor": account,
            "limit": 10  # Get latest 10 posts per account
        }
        
        response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
        
        if response.status_code == 200:
            data = response.json()
            feed_items = data.get('feed', [])
            
            for item in feed_items:
                try:
                    post = item.get('post', {})
                    record = post.get('record', {})
                    
                    # Extract post content
                    text = record.get('text', '')
                    created_at = record.get('createdAt', '')
                    
                    # Skip if no text
                    if not text or len(text) < 20:
                        continue
                    
                    # Build post URL
                    post_uri = post.get('uri', '')
                    post_url = f"https://bsky.app/profile/{account}/post/{post_uri.split('/')[-1]}" if post_uri else ''
                    
                    # Parse timestamp
                    try:
                        pub_date = datetime.strptime(created_at, '%Y-%m-%dT%H:%M:%S.%fZ')
                    except:
                        pub_date = datetime.now()
                    
                    # Extract team tags from text
                    tags = ['NFL']  # Default
                    text_lower = text.lower()
                    for team_abbr in ['BUF', 'MIA', 'NE', 'NYJ', 'BAL', 'CIN', 'CLE', 'PIT',
                                     'HOU', 'IND', 'JAX', 'TEN', 'DEN', 'KC', 'LV', 'LAC',
                                     'DAL', 'NYG', 'PHI', 'WAS', 'CHI', 'DET', 'GB', 'MIN',
                                     'ATL', 'CAR', 'NO', 'TB', 'ARI', 'LAR', 'SF', 'SEA']:
                        if team_abbr.lower() in text_lower:
                            tags.append(team_abbr)
                    
                    article = {
                        "article_id": generate_hash(post_url or text),
                        "source_name": f"bluesky_{account.split('.')[0]}",
                        "source_type": "bluesky_api",
                        "author_name": account,
                        "title": text[:100] + '...' if len(text) > 100 else text,
                        "summary": text[:500],
                        "full_text": text,
                        "article_url": post_url or f"https://bsky.app/profile/{account}",
                        "published_at": pub_date,
                        "ingested_at": datetime.now(),
                        "rss_feed_url": f"{BLUESKY_API_BASE}/xrpc/app.bsky.feed.getAuthorFeed?actor={account}",
                        "tags": tags,
                        "content_hash": generate_hash(text),
                        "is_processed": False,
                        "processed_at": None,
                        "created_at": datetime.now(),
                        "updated_at": datetime.now()
                    }
                    articles_list.append(article)
                    bluesky_posts_fetched += 1
                except Exception as e:
                    continue
            
            bluesky_accounts_success += 1
            print(f"  ✓ {account}: {len(feed_items)} posts")
        else:
            print(f"  ✗ {account}: HTTP {response.status_code}")
        
        # Rate limiting
        time.sleep(0.3)
        
    except Exception as e:
        print(f"  ✗ {account}: Error - {str(e)[:50]}")

print(f"\n  📊 Bluesky: {bluesky_posts_fetched} posts from {bluesky_accounts_success}/{len(BLUESKY_ACCOUNTS)} accounts")

# === Scrape NFL Team Official Websites (All 32) ===
print("\n🏈 Scraping NFL team official websites...")

team_articles_total = 0
team_sites_success = 0
team_sites_failed = 0

for team_abbr, url in NFL_TEAM_NEWS_URLS.items():
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find all links that look like news articles
            all_links = soup.find_all('a', href=True)
            team_news_items = []
            seen_urls = set()
            
            for link in all_links:
                href = link.get('href', '')
                text = link.get_text(strip=True)
                
                # Skip if no meaningful text
                if len(text) < 20:
                    continue
                
                # Look for news article patterns in URL
                if '/news/' in href or '/articles/' in href:
                    # Make absolute URL
                    if not href.startswith('http'):
                        if href.startswith('/'):
                            # Extract base domain from url
                            from urllib.parse import urlparse
                            parsed = urlparse(url)
                            base_url = f"{parsed.scheme}://{parsed.netloc}"
                            href = f"{base_url}{href}"
                        else:
                            continue
                    
                    # Skip duplicates
                    if href in seen_urls:
                        continue
                    
                    seen_urls.add(href)
                    team_news_items.append({
                        'url': href,
                        'title': text,
                        'team': team_abbr
                    })
                    
                    # Limit per team
                    if len(team_news_items) >= MAX_TEAM_ARTICLES_PER_SITE:
                        break
            
            # Add to articles list
            for article_data in team_news_items:
                article = {
                    "article_id": generate_hash(article_data['url']),
                    "source_name": f"{team_abbr.lower()}_official",
                    "source_type": "team_website",
                    "author_name": f"{team_abbr} Official",
                    "title": article_data['title'],
                    "summary": article_data['title'],
                    "full_text": article_data['title'],
                    "article_url": article_data['url'],
                    "published_at": datetime.now(),
                    "ingested_at": datetime.now(),
                    "rss_feed_url": url,
                    "tags": [team_abbr],
                    "content_hash": generate_hash(article_data['title']),
                    "is_processed": False,
                    "processed_at": None,
                    "created_at": datetime.now(),
                    "updated_at": datetime.now()
                }
                articles_list.append(article)
            
            team_articles_total += len(team_news_items)
            team_sites_success += 1
            
            if len(team_news_items) > 0:
                print(f"  ✓ {team_abbr}: {len(team_news_items)} articles")
        else:
            team_sites_failed += 1
            
        # Rate limiting between teams
        time.sleep(0.2)
        
    except Exception as e:
        team_sites_failed += 1
        continue

print(f"\n  📊 Team Sites: {team_articles_total} articles from {team_sites_success}/32 teams ({team_sites_failed} failed)")

# === Scrape FootballGuys.com News ===
print("\n🏈 Scraping FootballGuys.com news...")

try:
    url = "https://www.footballguys.com/news.php"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # FootballGuys has multiple potential structures - try them all
        footballguys_articles = []
        seen_urls = set()
        seen_titles = set()
        
        # Strategy 1: Find all links and filter for news-like content
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            try:
                href = link.get('href', '')
                text = link.get_text(strip=True)
                
                # Skip if no meaningful text
                if len(text) < 20:
                    continue
                
                # Skip navigation/header links
                if any(skip in text.lower() for skip in ['home', 'login', 'subscribe', 'menu', 'search', 'privacy']):
                    continue
                
                # Skip if duplicate title
                if text in seen_titles:
                    continue
                
                # Make absolute URL
                if href and not href.startswith('http'):
                    if href.startswith('/'):
                        href = f"https://www.footballguys.com{href}"
                    elif href.startswith('#') or href.startswith('javascript'):
                        continue
                    else:
                        href = f"https://www.footballguys.com/{href}"
                
                # Skip if duplicate URL
                if href in seen_urls:
                    continue
                
                # Look for player/team names or fantasy keywords to identify real news
                is_news = False
                text_lower = text.lower()
                
                # Check for fantasy keywords
                fantasy_keywords = ['injury', 'start', 'sit', 'week', 'fantasy', 'outlook', 'update', 
                                   'practice', 'status', 'questionable', 'doubtful', 'out', 'active']
                if any(kw in text_lower for kw in fantasy_keywords):
                    is_news = True
                
                # Check for team abbreviations
                for team_abbr in ['BUF', 'MIA', 'NE', 'NYJ', 'BAL', 'CIN', 'CLE', 'PIT',
                                 'HOU', 'IND', 'JAX', 'TEN', 'DEN', 'KC', 'LV', 'LAC',
                                 'DAL', 'NYG', 'PHI', 'WAS', 'CHI', 'DET', 'GB', 'MIN',
                                 'ATL', 'CAR', 'NO', 'TB', 'ARI', 'LAR', 'SF', 'SEA']:
                    if team_abbr.lower() in text_lower or team_abbr in text:
                        is_news = True
                        break
                
                # Check for position abbreviations
                if any(pos in text for pos in ['QB', 'RB', 'WR', 'TE', 'DST', 'K']):
                    is_news = True
                
                if not is_news:
                    continue
                
                seen_titles.add(text)
                seen_urls.add(href)
                
                # Extract team tags from title
                tags = ['Fantasy']
                for team_abbr in ['BUF', 'MIA', 'NE', 'NYJ', 'BAL', 'CIN', 'CLE', 'PIT',
                                 'HOU', 'IND', 'JAX', 'TEN', 'DEN', 'KC', 'LV', 'LAC',
                                 'DAL', 'NYG', 'PHI', 'WAS', 'CHI', 'DET', 'GB', 'MIN',
                                 'ATL', 'CAR', 'NO', 'TB', 'ARI', 'LAR', 'SF', 'SEA']:
                    if team_abbr.lower() in text_lower or team_abbr in text:
                        tags.append(team_abbr)
                
                # Get parent element for more context
                parent = link.parent
                summary_text = text
                if parent:
                    parent_text = parent.get_text(strip=True)
                    if len(parent_text) > len(text):
                        summary_text = parent_text[:500]
                
                footballguys_articles.append({
                    'url': href,
                    'title': text,
                    'summary': summary_text,
                    'tags': tags
                })
                
                # Limit to avoid infinite growth
                if len(footballguys_articles) >= 50:
                    break
                    
            except Exception as e:
                continue
        
        # Add to articles list
        for article_data in footballguys_articles:
            article = {
                "article_id": generate_hash(article_data['url']),
                "source_name": "footballguys_com",
                "source_type": "footballguys_scrape",
                "author_name": "FootballGuys.com",
                "title": article_data['title'],
                "summary": article_data['summary'],
                "full_text": article_data['summary'],
                "article_url": article_data['url'],
                "published_at": datetime.now(),
                "ingested_at": datetime.now(),
                "rss_feed_url": url,
                "tags": article_data['tags'],
                "content_hash": generate_hash(article_data['title']),
                "is_processed": False,
                "processed_at": None,
                "created_at": datetime.now(),
                "updated_at": datetime.now()
            }
            articles_list.append(article)
        
        print(f"  ✓ FootballGuys: {len(footballguys_articles)} articles scraped")
    else:
        print(f"  ✗ FootballGuys: HTTP {response.status_code}")
        
except Exception as e:
    print(f"  ✗ FootballGuys: Error - {str(e)[:50]}")

# === Scrape NFL.com Main News Page ===
print("\n🏈 Scraping NFL.com main news page...")

try:
    url = "https://www.nfl.com/news"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find all article links
        all_links = soup.find_all('a', href=True)
        article_links = []
        
        for link in all_links:
            href = link.get('href', '')
            text = link.get_text(strip=True)
            
            # Filter for actual news articles
            if '/news/' in href and text and len(text) > 20:
                # Make absolute URL
                if href.startswith('/'):
                    href = f"https://www.nfl.com{href}"
                
                # Extract potential team tags from URL or text
                tags = []
                for team_abbr in ['BUF', 'MIA', 'NE', 'NYJ', 'BAL', 'CIN', 'CLE', 'PIT',
                                 'HOU', 'IND', 'JAX', 'TEN', 'DEN', 'KC', 'LV', 'LAC',
                                 'DAL', 'NYG', 'PHI', 'WAS', 'CHI', 'DET', 'GB', 'MIN',
                                 'ATL', 'CAR', 'NO', 'TB', 'ARI', 'LAR', 'SF', 'SEA']:
                    if team_abbr.lower() in text.lower() or team_abbr.lower() in href.lower():
                        tags.append(team_abbr)
                
                article_links.append({
                    'url': href,
                    'title': text,
                    'tags': tags if tags else ['NFL']
                })
        
        # Deduplicate by URL and limit to 30 articles
        seen_urls = set()
        unique_articles = []
        for article in article_links:
            if article['url'] not in seen_urls:
                seen_urls.add(article['url'])
                unique_articles.append(article)
                if len(unique_articles) >= 30:
                    break
        
        # Add to articles list
        for article_data in unique_articles:
            article = {
                "article_id": generate_hash(article_data['url']),
                "source_name": "nfl_com_news",
                "source_type": "nfl_com_scrape",
                "author_name": "NFL.com",
                "title": article_data['title'],
                "summary": article_data['title'],  # We only have title from listing page
                "full_text": article_data['title'],
                "article_url": article_data['url'],
                "published_at": datetime.now(),
                "ingested_at": datetime.now(),
                "rss_feed_url": url,
                "tags": article_data['tags'],
                "content_hash": generate_hash(article_data['title']),
                "is_processed": False,
                "processed_at": None,
                "created_at": datetime.now(),
                "updated_at": datetime.now()
            }
            articles_list.append(article)
        
        print(f"  ✓ NFL.com: {len(unique_articles)} articles scraped")
    else:
        print(f"  ✗ NFL.com: HTTP {response.status_code}")
        
except Exception as e:
    print(f"  ✗ NFL.com: Error - {str(e)[:50]}")

print(f"\n📊 Total articles fetched: {len(articles_list)}")

# === Write to Unity Catalog with Deduplication ===
if len(articles_list) > 0:
    print("\n💾 Writing to main.fantasai_news.raw_rss_articles...")
    
    # Convert to DataFrame with explicit schema for tags
    articles_df = pd.DataFrame(articles_list)
    spark_df = spark.createDataFrame(articles_df, schema="""
        article_id STRING,
        source_name STRING,
        source_type STRING,
        author_name STRING,
        title STRING,
        summary STRING,
        full_text STRING,
        article_url STRING,
        published_at TIMESTAMP,
        ingested_at TIMESTAMP,
        rss_feed_url STRING,
        tags ARRAY<STRING>,
        content_hash STRING,
        is_processed BOOLEAN,
        processed_at TIMESTAMP,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    """)
    
    # Read existing article IDs for deduplication
    existing_ids = spark.sql("""
        SELECT article_id FROM main.fantasai_news.raw_rss_articles
    """).toPandas()['article_id'].tolist()
    
    # Filter out duplicates
    new_articles_df = spark_df.filter(~F.col('article_id').isin(existing_ids))
    new_count = new_articles_df.count()
    
    if new_count > 0:
        new_articles_df.write.mode('append').saveAsTable('main.fantasai_news.raw_rss_articles')
        print(f"  ✅ Inserted {new_count} new articles")
    else:
        print(f"  ℹ️  No new articles (all {len(articles_list)} were duplicates)")
else:
    print("\n⚠️  No articles fetched")

phase1_time = time.time() - start_time
print(f"\n⏱️  Phase 1 completed in {phase1_time:.1f} seconds")
print("=" * 70)

In [0]:
print("\n" + "=" * 70)
print("🔍 PHASE 2: PLAYER ENTITY EXTRACTION")
print("=" * 70)

import pytz
start_time = time.time()

# === Load Player Roster ===
print("\n📋 Loading player roster from Unity Catalog...")

try:
    player_roster = spark.sql("""
        SELECT DISTINCT
            s.player_id,
            s.player_name,
            s.position,
            s.team
        FROM main.fantasai.silver_weekly_stats s
        WHERE s.season = 2024
            AND s.position IN ('QB', 'RB', 'WR', 'TE')
            AND s.player_name IS NOT NULL
    """).toPandas()
    
    print(f"  ✓ Loaded {len(player_roster)} players")
except Exception as e:
    print(f"  ⚠️  Could not load player roster: {str(e)[:100]}")
    print(f"  ℹ️  Skipping Phase 2 - player roster required")
    phase2_time = time.time() - start_time
    print(f"\n⏱️  Phase 2 skipped in {phase2_time:.1f} seconds")
    print("=" * 70)
    player_roster = pd.DataFrame()  # Empty dataframe

if len(player_roster) > 0:
    # === Load Unprocessed Articles ===
    print("\n📰 Loading unprocessed articles...")
    
    articles_to_process = spark.sql("""
        SELECT 
            article_id,
            source_name,
            title,
            COALESCE(full_text, summary) as content,
            article_url as source_url,
            published_at,
            tags
        FROM main.fantasai_news.raw_rss_articles
        WHERE is_processed = FALSE
            OR article_id NOT IN (
                SELECT DISTINCT source_id 
                FROM main.fantasai_news.enriched_news
            )
        ORDER BY published_at DESC
        LIMIT 50
    """).toPandas()
    
    print(f"  ✓ Found {len(articles_to_process)} articles to process")
    
    # === Extract Player Mentions ===
    if len(articles_to_process) > 0:
        print("\n🔎 Extracting player mentions...")
        
        enriched_articles = []
        utc = pytz.UTC
        
        for idx, article in articles_to_process.iterrows():
            text = f"{article['title']} {article['content']}".lower()
            mentioned_players = []
            
            for _, player in player_roster.iterrows():
                player_name_lower = player['player_name'].lower()
                
                # Simple pattern matching
                if player_name_lower in text:
                    # Calculate confidence based on name uniqueness
                    name_parts = player_name_lower.split()
                    if len(name_parts) >= 2:
                        confidence = 0.9 if len(name_parts) == 2 else 0.8
                    else:
                        confidence = 0.5
                    
                    mentioned_players.append({
                        'player_id': str(player['player_id']),
                        'player_name': str(player['player_name']),
                        'position': str(player['position']),
                        'team': str(player['team']),
                        'confidence': float(confidence)
                    })
            
            if len(mentioned_players) > 0:
                # Sort by confidence and take top player as primary
                mentioned_players.sort(key=lambda x: x['confidence'], reverse=True)
                
                # Handle tags array properly
                tags_value = article['tags']
                if tags_value is None:
                    mentioned_teams = []
                elif isinstance(tags_value, list):
                    mentioned_teams = tags_value
                else:
                    mentioned_teams = []
                
                # Convert pandas timestamp to python datetime with timezone
                pub_ts = article['published_at']
                if isinstance(pub_ts, pd.Timestamp):
                    if pub_ts.tzinfo is None:
                        pub_dt = pub_ts.to_pydatetime().replace(tzinfo=utc)
                    else:
                        pub_dt = pub_ts.to_pydatetime()
                else:
                    pub_dt = datetime.now(utc)
                
                now_utc = datetime.now(utc)
                
                enriched_articles.append({
                    'news_id': generate_hash(article['article_id'] + str(now_utc)),
                    'source_id': str(article['article_id']),
                    'source_table': 'main.fantasai_news.raw_rss_articles',
                    'headline': str(article['title']),
                    'full_text': str(article['content']),
                    'source_url': str(article['source_url']),
                    'published_at': pub_dt,
                    'mentioned_players': mentioned_players,
                    'primary_player_id': str(mentioned_players[0]['player_id']),
                    'mentioned_teams': mentioned_teams,
                    'entity_extraction_model': 'regex_pattern_match',
                    'extraction_confidence': float(mentioned_players[0]['confidence']),
                    'enriched_at': now_utc,
                    'created_at': now_utc,
                    'updated_at': now_utc
                })
        
        print(f"  ✓ Extracted {len(enriched_articles)} articles with player mentions")
        
        # === Write to enriched_news ===
        if len(enriched_articles) > 0:
            print("\n💾 Writing to main.fantasai_news.enriched_news...")
            
            # Define explicit schema matching actual table
            from pyspark.sql.types import StructType, StructField, StringType, ArrayType, DoubleType, TimestampType
            
            player_mention_schema = StructType([
                StructField("player_id", StringType(), True),
                StructField("player_name", StringType(), True),
                StructField("position", StringType(), True),
                StructField("team", StringType(), True),
                StructField("confidence", DoubleType(), True)
            ])
            
            enriched_schema = StructType([
                StructField("news_id", StringType(), False),
                StructField("source_id", StringType(), False),
                StructField("source_table", StringType(), False),
                StructField("headline", StringType(), False),
                StructField("full_text", StringType(), False),
                StructField("source_url", StringType(), True),
                StructField("published_at", TimestampType(), False),
                StructField("mentioned_players", ArrayType(player_mention_schema), True),
                StructField("primary_player_id", StringType(), True),
                StructField("mentioned_teams", ArrayType(StringType()), True),
                StructField("entity_extraction_model", StringType(), True),
                StructField("extraction_confidence", DoubleType(), True),
                StructField("enriched_at", TimestampType(), False),
                StructField("created_at", TimestampType(), True),
                StructField("updated_at", TimestampType(), True)
            ])
            
            enriched_df = spark.createDataFrame(enriched_articles, schema=enriched_schema)
            enriched_df.write.mode('append').saveAsTable('main.fantasai_news.enriched_news')
            
            # Update is_processed flag
            processed_ids = [a['source_id'] for a in enriched_articles]
            for article_id in processed_ids:
                spark.sql(f"""
                    UPDATE main.fantasai_news.raw_rss_articles
                    SET is_processed = TRUE,
                        processed_at = current_timestamp()
                    WHERE article_id = '{article_id}'
                """)
            
            print(f"  ✅ Inserted {len(enriched_articles)} enriched articles")
            print(f"  ✅ Updated {len(processed_ids)} articles as processed")
        else:
            print("  ℹ️  No articles with player mentions found")
    else:
        print("  ℹ️  No unprocessed articles found")

phase2_time = time.time() - start_time
print(f"\n⏱️  Phase 2 completed in {phase2_time:.1f} seconds")
print("=" * 70)

In [0]:
print("\n" + "=" * 70)
print("🤖 PHASE 3: AI SUMMARIZATION")
print("=" * 70)

start_time = time.time()

if llm_client is None:
    print("\n❌ LLM client not initialized. Skipping AI summarization.")
else:
    # === Load Articles to Summarize ===
    print("\n📰 Loading enriched articles for summarization...")
    
    to_summarize = spark.sql("""
        SELECT 
            news_id,
            headline,
            full_text,
            source_url,
            published_at,
            mentioned_players,
            primary_player_id,
            mentioned_teams
        FROM main.fantasai_news.enriched_news
        WHERE news_id NOT IN (
            SELECT DISTINCT news_id 
            FROM main.fantasai_news.ai_summaries
        )
        ORDER BY extraction_confidence DESC, published_at DESC
        LIMIT 50
    """).toPandas()
    
    print(f"  ✓ Found {len(to_summarize)} articles to summarize")
    
    if len(to_summarize) > 0:
        print(f"\n🔮 Generating AI summaries using {LLM_MODEL}...")
        
        # Fantasy football analyst prompt - more explicit about JSON format
        SYSTEM_PROMPT = """You are an expert fantasy football analyst. Analyze news articles and provide concise, actionable fantasy insights.
        
You MUST respond with ONLY a valid JSON object (no markdown, no code blocks, no explanatory text). Format:
{
  "summary_text": "2-3 sentence summary",
  "fantasy_insight": "Specific advice",
  "fantasy_relevance_score": 75,
  "impact_category": "injury",
  "priority_level": "high",
  "impacted_players": [{"player_name": "Name", "impact_direction": "negative"}],
  "is_time_sensitive": true
}

Categories: injury, opportunity, depth_chart, contract
Priority: critical, high, medium, low
Direction: positive, negative, neutral"""
        
        summaries = []
        total_tokens = 0
        
        for idx, article in to_summarize.iterrows():
            try:
                # Get player names safely - handle numpy arrays properly
                player_names = []
                mentioned = article['mentioned_players']
                if mentioned is not None and hasattr(mentioned, '__len__') and len(mentioned) > 0:
                    player_names = [p['player_name'] for p in mentioned[:3]]
                
                players_text = ', '.join(player_names) if player_names else 'None'
                
                user_prompt = f"""Article: {article['headline']}
                
Content: {article['full_text'][:1000]}
                
Players mentioned: {players_text}
                
Analyze for fantasy football implications. Return ONLY the JSON object."""
                
                response = llm_client.predict(
                    endpoint=LLM_MODEL,
                    inputs={
                        "messages": [
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": user_prompt}
                        ],
                        "max_tokens": 800,
                        "temperature": 0.3
                    }
                )
                
                # Parse JSON response with improved extraction
                content = response['choices'][0]['message']['content'].strip()
                
                # Try multiple extraction strategies
                json_text = None
                
                # Strategy 1: Remove markdown code blocks
                if '```json' in content:
                    json_text = content.split('```json')[1].split('```')[0].strip()
                elif '```' in content:
                    json_text = content.split('```')[1].split('```')[0].strip()
                # Strategy 2: Look for JSON object boundaries
                elif '{' in content and '}' in content:
                    start = content.find('{')
                    end = content.rfind('}') + 1
                    json_text = content[start:end]
                else:
                    json_text = content
                
                # Try to parse
                try:
                    ai_result = json.loads(json_text)
                except json.JSONDecodeError as je:
                    # Log the problematic response for debugging
                    print(f"  ⚠️  Article {idx + 1} JSON parse error. Response preview: {content[:200]}")
                    raise
                
                # Normalize impacted_players to match table schema
                impacted_players = ai_result.get('impacted_players', [])
                normalized_players = []
                for player in impacted_players:
                    normalized_players.append({
                        'player_id': None,
                        'player_name': player.get('player_name', ''),
                        'impact_direction': player.get('impact_direction', 'neutral'),
                        'impact_magnitude': None
                    })
                
                summaries.append({
                    'summary_id': generate_hash(article['news_id'] + str(datetime.now())),
                    'news_id': article['news_id'],
                    'summary_text': ai_result.get('summary_text', ''),
                    'fantasy_insight': ai_result.get('fantasy_insight', ''),
                    'fantasy_relevance_score': float(ai_result.get('fantasy_relevance_score', 50)),
                    'impact_category': ai_result.get('impact_category', 'unknown'),
                    'priority_level': ai_result.get('priority_level', 'medium'),
                    'impacted_players': normalized_players,
                    'is_time_sensitive': ai_result.get('is_time_sensitive', False),
                    'llm_model': LLM_MODEL,
                    'llm_tokens_used': response.get('usage', {}).get('total_tokens', 0),
                    'generated_at': datetime.now()
                })
                
                total_tokens += response.get('usage', {}).get('total_tokens', 0)
                
                if (idx + 1) % 10 == 0:
                    print(f"  Progress: {idx + 1}/{len(to_summarize)} articles processed")
                    time.sleep(1)
                
            except Exception as e:
                print(f"  ✗ Error processing article {idx + 1}: {str(e)[:150]}")
                continue
        
        print(f"\n  ✓ Generated {len(summaries)} AI summaries (success rate: {len(summaries)}/{len(to_summarize)})")
        print(f"  🪙 Total tokens used: {total_tokens:,}")
        
        # === Write to ai_summaries ===
        if len(summaries) > 0:
            print("\n💾 Writing to main.fantasai_news.ai_summaries...")
            
            from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType, IntegerType, ArrayType
            
            schema = StructType([
                StructField('summary_id', StringType(), False),
                StructField('news_id', StringType(), False),
                StructField('summary_text', StringType(), False),
                StructField('fantasy_insight', StringType(), True),
                StructField('fantasy_relevance_score', DoubleType(), True),
                StructField('impact_category', StringType(), True),
                StructField('priority_level', StringType(), True),
                StructField('impacted_players', ArrayType(StructType([
                    StructField('player_id', StringType(), True),
                    StructField('player_name', StringType(), True),
                    StructField('impact_direction', StringType(), True),
                    StructField('impact_magnitude', DoubleType(), True)
                ])), True),
                StructField('is_time_sensitive', BooleanType(), True),
                StructField('llm_model', StringType(), True),
                StructField('llm_tokens_used', IntegerType(), True),
                StructField('generated_at', TimestampType(), False)
            ])
            
            summaries_df = spark.createDataFrame(summaries, schema=schema)
            summaries_df.write.mode('append').saveAsTable('main.fantasai_news.ai_summaries')
            
            print(f"  ✅ Inserted {len(summaries)} AI summaries")
        else:
            print("  ⚠️  No summaries generated successfully")
    else:
        print("  ℹ️  No articles to summarize")

phase3_time = time.time() - start_time
print(f"\n⏱️  Phase 3 completed in {phase3_time:.1f} seconds")
print("=" * 70)

In [0]:
print("\n" + "=" * 70)
print("📋 PHASE 4: PLAYER NOTES AGGREGATION")
print("=" * 70)

start_time = time.time()

# === Aggregate AI Summaries by Player ===
print("\n🔄 Aggregating AI summaries by player...")

player_aggregates = spark.sql("""
    WITH player_mentions AS (
        -- Explode impacted_players array from AI summaries
        SELECT 
            ai.news_id,
            ai.summary_text,
            ai.fantasy_insight,
            ai.fantasy_relevance_score,
            ai.impact_category,
            ai.priority_level,
            ai.is_time_sensitive,
            ai.generated_at,
            en.source_url,
            en.published_at,
            -- Extract player info from impacted_players array
            explode(ai.impacted_players) as player_impact
        FROM main.fantasai_news.ai_summaries ai
        INNER JOIN main.fantasai_news.enriched_news en
            ON ai.news_id = en.news_id
    ),
    player_with_roster AS (
        -- Join with player roster to get player_id
        SELECT 
            pm.*,
            COALESCE(m.gsis_id, s.player_id) as player_id,
            s.position,
            s.team
        FROM player_mentions pm
        LEFT JOIN main.fantasai.silver_weekly_stats s
            ON LOWER(pm.player_impact.player_name) = LOWER(s.player_name)
            AND s.season = 2024
        LEFT JOIN main.fantasai.player_id_mapping m
            ON LOWER(s.player_name) = LOWER(m.full_name)
            AND m.season = 2024
        WHERE s.player_name IS NOT NULL
    ),
    aggregated_notes AS (
        SELECT 
            player_id,
            player_impact.player_name as player_name,
            position,
            team,
            -- Aggregate notes (latest 5)
            collect_list(
                struct(
                    news_id as note_id,
                    concat(summary_text, ' ', fantasy_insight) as note_text,
                    impact_category as impact_type,
                    player_impact.impact_direction,
                    priority_level as priority,
                    source_url,
                    published_at,
                    is_time_sensitive
                )
            ) as notes,
            -- Calculate aggregates
            AVG(fantasy_relevance_score) as overall_impact_score,
            -- Determine overall sentiment
            CASE 
                WHEN SUM(CASE WHEN player_impact.impact_direction = 'positive' THEN 1 ELSE 0 END) > 
                     SUM(CASE WHEN player_impact.impact_direction = 'negative' THEN 1 ELSE 0 END)
                    THEN 'positive'
                WHEN SUM(CASE WHEN player_impact.impact_direction = 'negative' THEN 1 ELSE 0 END) > 
                     SUM(CASE WHEN player_impact.impact_direction = 'positive' THEN 1 ELSE 0 END)
                    THEN 'negative'
                ELSE 'neutral'
            END as overall_sentiment,
            -- Flags
            MAX(CASE WHEN priority_level = 'critical' THEN 1 ELSE 0 END) = 1 as has_critical_news,
            MAX(CASE WHEN impact_category = 'injury' THEN 1 ELSE 0 END) = 1 as has_injury_concern,
            MAX(CASE WHEN impact_category = 'opportunity' THEN 1 ELSE 0 END) = 1 as has_opportunity_change,
            COUNT(*) as note_count,
            MAX(generated_at) as last_updated
        FROM player_with_roster
        GROUP BY player_id, player_impact.player_name, position, team
    )
    SELECT 
        player_id,
        player_name,
        position,
        team,
        slice(notes, 1, 5) as notes,  -- Keep only top 5
        overall_sentiment,
        overall_impact_score,
        has_critical_news,
        has_injury_concern,
        has_opportunity_change,
        note_count,
        last_updated,
        current_timestamp() as created_at,
        current_timestamp() as updated_at
    FROM aggregated_notes
""").toPandas()

print(f"  ✓ Aggregated notes for {len(player_aggregates)} players")

# === Merge/Upsert to player_notes ===
if len(player_aggregates) > 0:
    print("\n💾 Upserting to main.fantasai_news.player_notes...")
    
    # Create temp view
    spark.createDataFrame(player_aggregates).createOrReplaceTempView('new_player_notes')
    
    # Merge logic
    spark.sql("""
        MERGE INTO main.fantasai_news.player_notes target
        USING new_player_notes source
        ON target.player_id = source.player_id
        WHEN MATCHED THEN UPDATE SET
            target.player_name = source.player_name,
            target.position = source.position,
            target.team = source.team,
            target.notes = source.notes,
            target.overall_sentiment = source.overall_sentiment,
            target.overall_impact_score = source.overall_impact_score,
            target.has_critical_news = source.has_critical_news,
            target.has_injury_concern = source.has_injury_concern,
            target.has_opportunity_change = source.has_opportunity_change,
            target.note_count = source.note_count,
            target.last_updated = source.last_updated,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"  ✅ Upserted {len(player_aggregates)} player notes")
else:
    print("  ℹ️  No new player notes to aggregate")

phase4_time = time.time() - start_time
print(f"\n⏱️  Phase 4 completed in {phase4_time:.1f} seconds")
print("=" * 70)

In [0]:
%sql
-- Pipeline metrics across all phases

WITH pipeline_metrics AS (
  SELECT 
    'raw_rss_articles' as table_name,
    COUNT(*) as total_rows,
    COUNT(CASE WHEN is_processed = TRUE THEN 1 END) as processed_rows,
    MAX(published_at) as latest_date
  FROM main.fantasai_news.raw_rss_articles
  
  UNION ALL
  
  SELECT 
    'enriched_news',
    COUNT(*),
    NULL,
    MAX(published_at)
  FROM main.fantasai_news.enriched_news
  
  UNION ALL
  
  SELECT 
    'ai_summaries',
    COUNT(*),
    NULL,
    MAX(generated_at)
  FROM main.fantasai_news.ai_summaries
  
  UNION ALL
  
  SELECT 
    'player_notes',
    COUNT(*),
    NULL,
    MAX(last_updated)
  FROM main.fantasai_news.player_notes
),
top_players AS (
  SELECT 
    player_name,
    position,
    team,
    note_count,
    overall_impact_score,
    overall_sentiment,
    has_critical_news,
    has_injury_concern,
    has_opportunity_change
  FROM main.fantasai_news.player_notes
  WHERE note_count > 0
  ORDER BY overall_impact_score DESC, last_updated DESC
  LIMIT 10
),
ai_stats AS (
  SELECT 
    COUNT(*) as total_summaries,
    AVG(fantasy_relevance_score) as avg_relevance,
    SUM(llm_tokens_used) as total_tokens,
    COUNT(CASE WHEN priority_level = 'critical' THEN 1 END) as critical_count,
    COUNT(CASE WHEN is_time_sensitive = TRUE THEN 1 END) as time_sensitive_count
  FROM main.fantasai_news.ai_summaries
  WHERE DATE(generated_at) = CURRENT_DATE()
)

-- Display results
SELECT '=== PIPELINE METRICS ===' as section, '' as details
UNION ALL
SELECT table_name, CONCAT('Total: ', total_rows, ' | Processed: ', COALESCE(processed_rows, 0), ' | Latest: ', DATE(latest_date))
FROM pipeline_metrics

UNION ALL
SELECT '', ''

UNION ALL
SELECT '=== AI SUMMARY STATS (TODAY) ===' as section, '' as details
UNION ALL
SELECT 'Total Summaries', CAST(total_summaries AS STRING) FROM ai_stats
UNION ALL
SELECT 'Avg Relevance Score', CAST(ROUND(avg_relevance, 1) AS STRING) FROM ai_stats
UNION ALL
SELECT 'Total Tokens Used', CAST(total_tokens AS STRING) FROM ai_stats
UNION ALL
SELECT 'Critical News Count', CAST(critical_count AS STRING) FROM ai_stats
UNION ALL
SELECT 'Time-Sensitive Count', CAST(time_sensitive_count AS STRING) FROM ai_stats

UNION ALL
SELECT '', ''

UNION ALL
SELECT '=== TOP 10 PLAYERS WITH NEWS ===' as section, '' as details
UNION ALL
SELECT 
  player_name,
  CONCAT(
    position, '-', team, ' | ',
    note_count, ' notes | ',
    'Score: ', ROUND(overall_impact_score, 0), ' | ',
    overall_sentiment,
    CASE WHEN has_critical_news THEN ' 🚨' ELSE '' END,
    CASE WHEN has_injury_concern THEN ' 🩹' ELSE '' END,
    CASE WHEN has_opportunity_change THEN ' 📈' ELSE '' END
  )
FROM top_players

# ✅ Fantasy News Pipeline Complete!

---

## Summary

The daily news refresh pipeline has successfully updated all fantasy football intelligence tables:

### ✓ Phase 1: News Ingestion
Fetched and deduplicated articles from fantasy news APIs and NFL team RSS feeds

### ✓ Phase 2: Player Entity Extraction
Identified player mentions in news articles with confidence scoring

### ✓ Phase 3: AI Summarization
Generated fantasy insights using Databricks Foundation Model (Llama 3.3 70B)

### ✓ Phase 4: Player Notes Aggregation
Created consolidated player notes with impact scores and sentiment analysis

---

## 📊 View Results Above

Check the **Pipeline Verification** cell above to see:
- Article counts by phase
- AI summary statistics
- Top 10 players with recent news
- Critical alerts and injury concerns

---

## 🔄 Next Run

This pipeline runs daily at **7:00 AM Central Time**  
Next scheduled execution: Tomorrow morning

---

## 📱 API Access

Player notes and AI summaries are available in:
- `main.fantasai_news.player_notes` - Aggregated player intelligence
- `main.fantasai_news.ai_summaries` - Detailed AI-generated insights
- `main.fantasai_news.enriched_news` - Player-tagged articles

---

**Pipeline maintained by:** FantasAI Platform Team  
**Last updated:** May 30, 2026